# 8.5 Sigma Sweep Full Run (6D z)
Full sigma sweep (0.1, 0.5, 1.0, 2.0, 5.0, 7.5, 10.0) x 5 seeds (0-4), all 4 models (`decipher`, `decipher_vz`, `decipher_vz2`, `decipher_mf`), with the 6D multivariate-normal batch-shift simulation and dim_z=6 for every model. Mirrors `8.3 sigma-sweep-4-models.ipynb`'s structure, importing from `sigma_sweep_6dz` instead of `sigma_sweep`.

## Import modules

In [ ]:
import sys
sys.path.insert(0, "/Users/alanxie/CodingProjects/Columbia/Azizi Lab/Spatial Nut Carcinoma/decipher-bc-methods/Data/Simulated Data/Simulated Data Generation")

from sigma_sweep_6dz import rho_for_run

## Create simulated data and train models
Sigma sweep at 0.1, 0.5, 1.0, 2.0, 5.0, 7.5, 10.0.   
Seeds at 0, 1, 2, 3, 4

In [ ]:
import os
import pandas as pd

results_dir = "/Users/alanxie/CodingProjects/Columbia/Azizi Lab/Spatial Nut Carcinoma/decipher-bc-methods/Data/Simulated Data/Simulated Adata/6dz_shift_sigma_sweep/trained"
os.makedirs(results_dir, exist_ok=True)

models = {
    "Base Decipher": "decipher",
    "Decipher-VZ": "decipher_vz",
    "Decipher-VZ2": "decipher_vz2",
    "Decipher-MF": "decipher_mf",
}

sigmas = [0.1, 0.5, 1.0, 2.0, 5.0, 7.5, 10.0]
seeds = [0, 1, 2, 3, 4]

records = []
for model_name, model_tag in models.items():
    for sigma in sigmas:
        for seed in seeds:
            rho, gt_path, trained_path, adata = rho_for_run(sigma=sigma, seed=seed, model=model_tag, decipher_seed=1)
            records.append({
                "model": model_name,
                "sigma": sigma,
                "seed": seed,
                "rho": rho,
                "ground_truth_h5ad": gt_path,
                "trained_h5ad": trained_path,
            })

results_df = pd.DataFrame(records)
results_df.to_csv("sigma_sweep_results_6dz.csv", index=False)
results_df

## Correlation Plot

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

colors = {
    "Decipher-VZ": "#2E7D32",
    "Decipher-VZ2": "#1565C0",
    "Decipher-MF": "#F9A825",
    "Base Decipher": "#C62828",
}

sigmas_sorted = sorted(results_df["sigma"].unique())
x_pos = np.arange(len(sigmas_sorted))  # evenly spaced positions, one per sigma tested

fig, ax = plt.subplots(figsize=(7, 5))
for name in models:
    sub = results_df[results_df["model"] == name]
    stats = sub.groupby("sigma")["rho"].agg(["mean", "std"]).reindex(sigmas_sorted)
    mean = stats["mean"].to_numpy()
    sd_ = stats["std"].to_numpy()
    ax.plot(x_pos, mean, marker="o", color=colors[name], label=name, linewidth=2)
    ax.fill_between(x_pos, mean - sd_, mean + sd_, color=colors[name], alpha=0.18)

ax.set_xticks(x_pos)
ax.set_xticklabels([str(s) for s in sigmas_sorted])
ax.set_xlabel(r"batch-shift noise  $\sigma$   (shifts $\sim \mathcal{N}(0,\sigma^2)$, 6D)")
ax.set_ylabel(r"Spearman $|\rho|$  (decipher_time vs latent_t)")
ax.set_title("Trajectory recovery vs batch noise (6D z)")
ax.set_ylim(0, 1); ax.axhline(0, color="0.8", lw=0.8)
ax.legend(frameon=False); ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
fig.savefig("sigma_vs_correlation_6dz.png", dpi=200)
plt.show()

In [ ]:
rho_summary = results_df.groupby("sigma")["rho"].agg(["mean", "std"]).reindex(sigmas_sorted).round(3)
rho_summary

## Z-space UMAP colored by batch, highest sigma
Qualitative check (per the advisor's ask): confirm the learned `decipher_z` UMAP shows batch separation at the highest sigma tested, for every model.

In [ ]:
import scanpy as sc
from sklearn.metrics import silhouette_score

def load_run(results_df, sigma, model_name):
    row = results_df[(results_df["sigma"] == sigma) & (results_df["model"] == model_name)].iloc[0]
    return row, sc.read_h5ad(row["trained_h5ad"])

In [ ]:
latent_z_cols = ["latent_z0", "latent_z1", "latent_z2", "latent_z3", "latent_z4", "latent_z5"]
sigma_max = max(sigmas_sorted)

for model_name in models:
    row, adata = load_run(results_df, sigma_max, model_name)
    batch_labels = adata.obs["batch"].astype(str)

    sil_ground_truth = silhouette_score(
        adata.obs[latent_z_cols].values, batch_labels,
        sample_size=min(5000, adata.n_obs), random_state=42,
    )
    sil_learned = silhouette_score(
        adata.obsm["decipher_z"], batch_labels,
        sample_size=min(5000, adata.n_obs), random_state=42,
    )
    print(f"{model_name}  sigma={sigma_max}  ground truth z silhouette (by batch) = {sil_ground_truth:.4f}")
    print(f"{model_name}  sigma={sigma_max}  learned decipher_z silhouette (by batch) = {sil_learned:.4f}")

    adata.obsm["latent_z"] = adata.obs[latent_z_cols].values
    sc.pp.neighbors(adata, use_rep="latent_z", key_added="latent_z_neighbors", random_state=42)
    sc.tl.umap(adata, neighbors_key="latent_z_neighbors", random_state=42)
    adata.obsm["latent_z_umap"] = adata.obsm["X_umap"]

    sc.pp.neighbors(adata, use_rep="decipher_z", random_state=42)
    sc.tl.umap(adata, random_state=42)
    adata.obsm["decipher_z_umap"] = adata.obsm["X_umap"]

    fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 4))
    sc.pl.embedding(adata, basis="latent_z_umap", color="batch",
                    ax=axes[0], show=False, title=f"Ground truth z (6D), UMAP ({model_name}, sigma={sigma_max})")
    sc.pl.embedding(adata, basis="decipher_z_umap", color="batch",
                    ax=axes[1], show=False, title=f"Learned decipher_z, UMAP ({model_name}, sigma={sigma_max})")
    plt.tight_layout()
    plt.show()